In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

# Load API keys
load_dotenv(".env")

# LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# Tool: Simple QA
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)
qa_tool = Tool(
    name="Simple QA",
    func=qa_chain.run,
    description="Answer factual questions clearly"
)

C:\Users\Paul\AppData\Local\Temp\ipykernel_22020\4182369516.py:18: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


![Model Diagram](memory.png)

In [3]:
#https://python.langchain.com/api_reference/langchain/memory.html?utm_source=chatgpt.com

In [4]:
#🧠 1. ConversationBufferMemory
from langchain.memory import ConversationBufferMemory

# Memory (stores chat history)
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True,handle_parsing_errors=True)

# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,# ZERO_SHOT_REACT_DESCRIPTION
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

#print(memory.chat_memory.messages)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

C:\Users\Paul\AppData\Local\Temp\ipykernel_22020\2901623225.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True,handle_parsing_errors=True)
C:\Users\Paul\AppData\Local\Temp\ipykernel_22020\2901623225.py:8: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-ag

1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: At its simplest, **LangChain is a software framework designed to make it easier to build applications powered by Large Language Models (LLMs)**, such as GPT-4, Claude, or Llama.

Think of an LLM as a brilliant brain that knows a lot of information but is "trapped" inside a static model. LangChain acts as the **connective tissue** that allows that brain to interact with the real world.

Here is how it works in three key ways:

### 1. It connects LLMs to external data
By default, an LLM only knows what it was trained on. LangChain allows you to connect an LLM to your own data sources—such as PDFs, databases, websites, or private company documents—so the model can answer questions based on your specific information (a process often called RAG: Retrieval-Augmented Generation).

### 2. It enables "Chaining"
LLMs are often best at co

In [5]:
#2)ConversationBufferWindowMemory

from langchain.memory import ConversationBufferWindowMemory
from langchain.agents import initialize_agent, AgentType

memory = ConversationBufferWindowMemory(k=3,memory_key="chat_history")

agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

1️⃣ First Question


> Entering new AgentExecutor chain...


C:\Users\Paul\AppData\Local\Temp\ipykernel_22020\3124292110.py:6: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=3,memory_key="chat_history")


Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: At its simplest, **LangChain is a software framework designed to make it easier to build applications powered by Large Language Models (LLMs)**, such as GPT-4, Claude, or Llama.

Think of an LLM as a brilliant brain that knows a lot of information but is "trapped" inside a static model. LangChain acts as the **connective tissue** that allows that brain to interact with the real world.

Here is how it works in three key ways:

### 1. It connects LLMs to external data
By default, an LLM only knows what it was trained on. LangChain allows you to connect an LLM to your own data sources—such as PDFs, databases, websites, or private company documents—so the model can answer questions based on your specific information (a process often called RAG: Retrieval-Augmented Generation).

### 2. It enables "Chaining"
LLMs are often best at complex tasks when they are broken down into smaller steps. L

In [10]:
#🧠 3. ConversationSummaryMemory
from langchain.memory import ConversationSummaryMemory
from langchain.chat_models import ChatOpenAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
memory = ConversationSummaryMemory(llm=llm,memory_key="chat_history")
# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)


# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)

for msg in memory.chat_memory.messages:
    print(f"{msg.type.upper()}: {msg.content}")

1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: At its simplest, **LangChain is a software framework designed to make it easier to build applications powered by Large Language Models (LLMs)**, such as GPT-4, Claude, or Llama.

Think of an LLM as a brilliant brain that knows a lot of information but is "trapped" inside a static model. LangChain acts as the **connective tissue** that allows that brain to interact with the real world.

Here is how it works in three key ways:

### 1. It connects LLMs to external data
By default, an LLM only knows what it was trained on. LangChain allows you to connect an LLM to your own data sources—such as PDFs, databases, websites, or private company documents—so the model can answer questions based on your specific information (a process often called RAG: Retrieval-Augmented Generation).

### 2. It enables "Chaining"
LLMs are often best at co

In [13]:
from langchain.memory import VectorStoreRetrieverMemory
from langchain.vectorstores import FAISS
#from langchain.embeddings import OpenAIEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings


# embedding = OpenAIEmbeddings()
embedding = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
retriever = FAISS.from_texts(["initial memory"], embedding).as_retriever()

memory = VectorStoreRetrieverMemory(retriever=retriever,memory_key="chat_history")
# Agent with tool + memory
agent = initialize_agent(
    tools=[qa_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory
)

# Run multiple interactions
print("1️⃣ First Question")
res1 = agent.run("What is LangChain?")
print("\nAnswer:", res1)

print("\n2️⃣ Follow-up Question")
res2 = agent.run("Who created it?")
print("\nAnswer:", res2)

print("\n3️⃣ Ask again about previous topic")
res3 = agent.run("Explain it simply again.")
print("\nAnswer:", res3)


C:\Users\Paul\AppData\Local\Temp\ipykernel_22020\1236679355.py:11: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(retriever=retriever,memory_key="chat_history")


1️⃣ First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: At its simplest, **LangChain is a software framework designed to make it easier to build applications powered by Large Language Models (LLMs)**, such as GPT-4, Claude, or Llama.

Think of an LLM as a brilliant brain that knows a lot of information but is "trapped" inside a static model. LangChain acts as the **connective tissue** that allows that brain to interact with the real world.

Here is how it works in three key ways:

### 1. It connects LLMs to external data
By default, an LLM only knows what it was trained on. LangChain allows you to connect an LLM to your own data sources—such as PDFs, databases, websites, or private company documents—so the model can answer questions based on your specific information (a process often called RAG: Retrieval-Augmented Generation).

### 2. It enables "Chaining"
LLMs are often best at co

In [14]:
# Check what's stored in FAISS retriever
docs = retriever.vectorstore.similarity_search("memory")
for i, doc in enumerate(docs):
    print(f"🔹 Document {i+1}: {doc.page_content}")

🔹 Document 1: initial memory
🔹 Document 2: input: What is LangChain?
output: LangChain is a powerful software framework designed to simplify the development of applications powered by Large Language Models (LLMs). Essentially, it acts as a bridge that connects these models to external data sources, enables complex workflows through "chaining" different tasks together, and allows for the creation of "agents" that can interact with the real world using tools. By providing a standardized toolkit, it saves developers from writing custom code for API integrations, memory management, and data handling, making it much easier to build sophisticated, context-aware AI applications.
🔹 Document 3: input: Explain it simply again.
output: Think of LangChain as a "connector" tool for AI. 

Imagine you have a super-smart AI assistant, but it doesn't know about your private files, can't check the live weather, and forgets what you said five minutes ago. LangChain provides the "plumbing" to hook that AI

In [15]:
# See all texts in FAISS index
print("All indexed texts:")
for i, doc in enumerate(retriever.vectorstore.docstore._dict.values()):
    print(f"📄 {i+1}: {doc.page_content}")


All indexed texts:
📄 1: initial memory
📄 2: input: What is LangChain?
output: LangChain is a powerful software framework designed to simplify the development of applications powered by Large Language Models (LLMs). Essentially, it acts as a bridge that connects these models to external data sources, enables complex workflows through "chaining" different tasks together, and allows for the creation of "agents" that can interact with the real world using tools. By providing a standardized toolkit, it saves developers from writing custom code for API integrations, memory management, and data handling, making it much easier to build sophisticated, context-aware AI applications.
📄 3: input: Who created it?
output: LangChain was created by Harrison Chase, who launched the project in October 2022 while he was working at Robust Intelligence.
📄 4: input: Explain it simply again.
output: Think of LangChain as a "connector" tool for AI. 

Imagine you have a super-smart AI assistant, but it doesn't

In [16]:
query = "LangChain founder"
results = memory.retriever.get_relevant_documents(query)

print(f"\n🧠 Retrieval for: '{query}'")
for i, doc in enumerate(results):
    print(f"🔹 {i+1}: {doc.page_content}")


C:\Users\Paul\AppData\Local\Temp\ipykernel_22020\1323272930.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = memory.retriever.get_relevant_documents(query)



🧠 Retrieval for: 'LangChain founder'
🔹 1: input: Who created it?
output: LangChain was created by Harrison Chase, who launched the project in October 2022 while he was working at Robust Intelligence.
🔹 2: input: What is LangChain?
output: LangChain is a powerful software framework designed to simplify the development of applications powered by Large Language Models (LLMs). Essentially, it acts as a bridge that connects these models to external data sources, enables complex workflows through "chaining" different tasks together, and allows for the creation of "agents" that can interact with the real world using tools. By providing a standardized toolkit, it saves developers from writing custom code for API integrations, memory management, and data handling, making it much easier to build sophisticated, context-aware AI applications.
🔹 3: input: Explain it simply again.
output: Think of LangChain as a "connector" tool for AI. 

Imagine you have a super-smart AI assistant, but it doesn't 

In [17]:
memory.save_context(
    {"input": "Who is the founder of LangChain?"},
    {"output": "Harrison Chase is the founder of LangChain."}
)


In [ ]:
#🧠 6. PostgresChatMessageHistory, RedisChatMessageHistory, etc.

from langchain.memory import ConversationBufferMemory
from langchain.storage import PostgresChatMessageHistory

history = PostgresChatMessageHistory(session_id="abc123", connection_string="postgresql://...")
memory = ConversationBufferMemory(chat_memory=history)